# 00. 環境チェック（段階0）

SageMaker Studio 上で、課題に必要な環境が動くかを確認する。

1. GPU と PyTorch
2. MuJoCo の描画バックエンド（EGL / OSMesa）
3. シーンとカメラ画像
4. シミュレーション速度（データ生成時間の見積もり）
5. ランダム動作の動画（合格ラインの確保）
6. SO-101 グリッパの開口幅
7. 真上から掴める作業領域
8. 小片（立方体）の把持テスト
9. 結果のまとめ（`summary.json`）

**前提**: `bash scripts/setup_sagemaker.sh` を実行済みで、カーネル「Python 3.12 (pai)」を選んでいること。

**所要時間の目安**: GPU 描画（EGL）なら数分、CPU 描画（OSMesa）なら 10 分程度。

出力は `outputs/00_env_check/` に保存される。最後の `summary.json` の内容と動画を共有してほしい。

In [ ]:
import json
import shutil
import subprocess
import sys
import time
from importlib.metadata import version
from pathlib import Path

REPO = Path.cwd().resolve()
if REPO.name == "notebooks":
    REPO = REPO.parent
OUT = REPO / "outputs" / "00_env_check"
OUT.mkdir(parents=True, exist_ok=True)

summary = {"python": sys.version.split()[0]}  # 最後に summary.json として保存する
print("repo  :", REPO)
print("python:", summary["python"])

## 1. GPU と PyTorch

GPU インスタンスなのに `cuda available: False` になる場合は、PyTorch の CUDA 版がドライバと合っていない。
`TORCH_BACKEND=cu126 bash scripts/setup_sagemaker.sh` のように指定して再セットアップする。

In [ ]:
import torch

gpu = None
if shutil.which("nvidia-smi"):
    smi = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True,
    )
    gpu = smi.stdout.strip() or None
print("nvidia-smi    :", gpu or "GPU なし")
print("torch         :", torch.__version__, "| CUDA build:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())

if torch.cuda.is_available():
    x = torch.randn(4096, 4096, device="cuda")
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(10):
        y = x @ x
    torch.cuda.synchronize()
    tflops = 10 * 2 * 4096**3 / (time.perf_counter() - t0) / 1e12
    print(f"行列積の速度  : {tflops:.1f} TFLOPS (fp32)")
elif gpu:
    print("⚠ GPU はあるが PyTorch から使えない。上の説明に従って再セットアップすること。")

try:
    from torchcodec.decoders import VideoDecoder  # noqa: F401  LeRobot が動画の読み込みに使う
    torchcodec_ok = True
except Exception as e:
    torchcodec_ok = False
    print("⚠ torchcodec を読み込めない:", type(e).__name__, "→ 学習時は --dataset.video_backend=pyav を指定")

summary.update(gpu=gpu, torch=torch.__version__, torch_cuda=torch.version.cuda,
               cuda_available=torch.cuda.is_available(), torchcodec_ok=torchcodec_ok)

## 2. 描画バックエンドの選択

MuJoCo は import 時に描画方式（`MUJOCO_GL`）を決めるため、**MuJoCo を import する前に**選ぶ。
EGL → OSMesa の順に試し描画し、OpenGL の描画器名（GL_RENDERER）から GPU で描画できているかも判定する。
EGL で描画できても、NVIDIA の EGL が見えない環境では CPU のソフトウェア描画（llvmpipe）になることがある。

やり直すときはカーネルを再起動してから実行すること。

In [ ]:
from so101_food.utils.gl import select_gl_backend

GL = select_gl_backend()
summary.update(gl_backend=GL.backend, gl_renderer=GL.renderer, gl_hardware=GL.hardware)
print("採用:", GL.backend, "|", "GPU 描画" if GL.hardware else "CPU 描画（遅い。データ生成の時間見積もりに注意）")

In [ ]:
import gymnasium as gym
import matplotlib.pyplot as plt
import mujoco
import numpy as np
import pandas as pd
import so101_nexus.mujoco  # noqa: F401  （MuJoCo 環境 ID の登録）
from IPython.display import Video, display
from so101_nexus import (
    CubeObject, JointPositions, OverheadCamera, PickConfig, RenderConfig, WristCamera,
)

from so101_food.utils.render import quality_for_gl, save_mp4

# GPU 描画なら so101-nexus 既定の高品質、CPU 描画なら影・アンチエイリアスなしの高速設定
QUALITY = quality_for_gl(GL)
summary.update(mujoco=mujoco.__version__, so101_nexus=version("so101-nexus"),
               lerobot=version("lerobot"), render_quality=repr(QUALITY))
print("mujoco", mujoco.__version__, "| so101-nexus", version("so101-nexus"),
      "| lerobot", version("lerobot"))
print("描画品質:", QUALITY)

# 小片を置く領域（ロボット前方。真上から掴める範囲に収める。7 節で確認）
SPAWN = dict(spawn_center=(0.20, 0.0), spawn_min_radius=0.0, spawn_max_radius=0.06)

## 3. シーンとカメラ画像

so101-nexus の `MuJoCoPickLift-v1`（物体を掴んで持ち上げるタスク）に 25mm 角の立方体を置き、
ポリシーの入力に使う俯瞰カメラ・手首カメラ（224×224）と、動画用の横からの視点を表示する。

In [ ]:
CAM = 224
cfg_cam = PickConfig(
    objects=CubeObject(side_length_mm=25, color="orange"),
    render=RenderConfig(quality=QUALITY, camera="side"),
    observations=[JointPositions(), WristCamera(width=CAM, height=CAM),
                  OverheadCamera(width=CAM, height=CAM)],
    **SPAWN,
)
env_cam = gym.make("MuJoCoPickLift-v1", config=cfg_cam, render_mode="rgb_array")
obs, info = env_cam.reset(seed=0)
print({k: v.shape for k, v in obs.items()}, "| task:", env_cam.unwrapped.task_description)

views = [("side view (render)", env_cam.render()), ("overhead camera", obs["overhead_camera"]),
         ("wrist camera", obs["wrist_camera"])]
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (title, img) in zip(axes, views):
    ax.imshow(img)
    ax.set_title(title)
    ax.axis("off")
fig.savefig(OUT / "cameras.png", dpi=100, bbox_inches="tight")
plt.show()

## 4. シミュレーション速度

データ生成時間の見積もりに使う。物理計算だけなら高速なので、段階1では
「エキスパートを描画なしで実行 → 成功したエピソードだけ描画し直す」構成にする予定。

In [ ]:
def ms_per_step(env, n):
    env.reset(seed=1)
    env.step(env.action_space.sample())
    t0 = time.perf_counter()
    for _ in range(n):
        env.step(env.action_space.sample())
    return (time.perf_counter() - t0) / n * 1e3


ms_cam = ms_per_step(env_cam, 50)
env_phys = gym.make("MuJoCoPickLift-v1", config=PickConfig(objects=CubeObject(side_length_mm=25)))
ms_phys = ms_per_step(env_phys, 300)
env_phys.close()

n_steps = 300 * 250  # 成功 300 エピソード × 250 ステップ
print(f"物理のみ          : {ms_phys:6.2f} ms/step")
print(f"2 カメラ({CAM}px)込み: {ms_cam:6.1f} ms/step")
print(f"→ 300 エピソード分の描画にかかる時間の目安: {n_steps * ms_cam / 1e3 / 60:.0f} 分")
summary.update(ms_per_step_physics=round(ms_phys, 3), ms_per_step_2cams=round(ms_cam, 1))

## 5. ランダム動作の動画（合格ラインの確保）

関節目標をランダムに動かすだけの 5 秒の動画。学習前のベースラインとしても使える。

In [ ]:
cfg_video = PickConfig(
    objects=CubeObject(side_length_mm=25, color="orange"),
    render=RenderConfig(quality=QUALITY, camera="side"),
    **SPAWN,
)
env_rand = gym.make("MuJoCoPickLift-v1", config=cfg_video, render_mode="rgb_array",
                    control_mode="pd_joint_target_delta_pos", max_episode_steps=10_000)
env_rand.reset(seed=0)
env_rand.action_space.seed(0)
frames = []
for t in range(250):  # 制御周期 0.02 秒 × 250 = 5 秒
    env_rand.step(env_rand.action_space.sample())
    if t % 2 == 0:  # 25 fps
        frames.append(env_rand.render())
env_rand.close()
path = save_mp4(frames, OUT / "random_actions.mp4", fps=25)
print("saved:", path.relative_to(REPO))
display(Video(str(path), embed=True, width=480))

## 6. SO-101 グリッパの開口幅

可動側の指がヒンジで回転するため、開口幅は関節角度と指先からの深さで変わる。

- `tip_gap_mm`: 両指先の中心間距離
- `pad_gap_mm`: 指の内側パッド同士の最短距離（挟める物体幅の目安）

計画メモの「最大辺はグリッパ開口幅の 7〜8 割以下」は、この `pad_gap_mm` を基準にする。

In [ ]:
from so101_food.sim.gripper import angle_for_gap, measure_gripper, samples_as_dicts, tcp_to_fingertip_mm

model = env_cam.unwrapped.model
samples = measure_gripper(model)
df_grip = pd.DataFrame(samples_as_dicts(samples))
df_grip.to_csv(OUT / "gripper_opening.csv", index=False)
display(df_grip.iloc[::2].set_index("angle_deg").T.round(1))

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(df_grip.angle_deg, df_grip.tip_gap_mm, label="tip gap")
ax.plot(df_grip.angle_deg, df_grip.pad_gap_mm, label="pad gap")
for w in (20, 30):
    ax.axhline(w, color="gray", lw=0.8, ls="--")
ax.set_xlabel("gripper joint angle [deg]")
ax.set_ylabel("opening [mm]")
ax.legend()
ax.grid(alpha=0.3)
fig.savefig(OUT / "gripper_opening.png", dpi=100, bbox_inches="tight")
plt.show()

angles = {w: round(angle_for_gap(samples, w), 1) for w in (20, 25, 30, 40)}
for w, a in angles.items():
    print(f"パッド間 {w} mm になる角度: {a:5.1f} deg")
print(f"関節範囲: {df_grip.angle_deg.min():.0f} 〜 {df_grip.angle_deg.max():.0f} deg"
      f"（最大開口 pad {df_grip.pad_gap_mm.max():.0f} mm）")
print(f"TCP から固定側指先まで: {tcp_to_fingertip_mm(model):.1f} mm")
summary["gripper"] = {
    "joint_range_deg": [float(df_grip.angle_deg.min()), float(df_grip.angle_deg.max())],
    "max_pad_gap_mm": round(float(df_grip.pad_gap_mm.max()), 1),
    "angle_deg_for_pad_gap_mm": angles,
    "tcp_to_fingertip_mm": round(tcp_to_fingertip_mm(model), 1),
}

## 7. 真上から掴める作業領域

TCP（両指の間）を真下に向けたまま届く範囲を IK で調べる。小片の向きはランダムなので、
閉じ方向 0°・45°・90°・135° のすべてで解けた点だけを「掴める」とする。

- 左: 把持の高さ（z = 12.5mm、25mm 角の中心）で真上姿勢をとれる範囲
- 右: 鍋の上での運搬を想定した高さ（z = 12cm）で、姿勢を問わず届く範囲

真上姿勢を保てるのは TCP の高さがおよそ 9cm まで（手首ピッチの可動域 ±95° の限界）なので、
鍋への運搬・投入では姿勢の条件を緩める必要がある。右図のベース近くの空白は、IK の初期値を
変えても解けなかった（高い位置ではベースに近づけない）。鍋はこの空白を避けて置く（位置は段階1で決める）。

In [ ]:
from matplotlib.patches import Circle, Rectangle

from so101_food.expert.ik import TopDownIK, reachability_grid

xs = np.arange(0.06, 0.401, 0.02)
ys = np.arange(-0.30, 0.301, 0.02)
t0 = time.perf_counter()
grid_grasp = reachability_grid(TopDownIK(model), xs, ys, z=0.0125,
                               yaws=(0.0, np.pi / 4, np.pi / 2, 3 * np.pi / 4))
grid_carry = reachability_grid(TopDownIK(model, rot_weight=0.02), xs, ys, z=0.12,
                               yaws=(0.0,), tilt_tol_deg=180.0)
print(f"計算時間: {time.perf_counter() - t0:.0f} 秒")

extent = (xs[0] - 0.01, xs[-1] + 0.01, ys[0] - 0.01, ys[-1] + 0.01)
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, grid, title in zip(axes, (grid_grasp, grid_carry),
                           ("top-down grasp @ z=12.5mm", "position only @ z=120mm")):
    ax.imshow(grid, origin="lower", extent=extent, cmap="Greens", vmin=0, vmax=1.5)
    ax.add_patch(Rectangle((0.14, -0.10), 0.12, 0.20, fill=False, ec="tab:orange", lw=2,
                           label="work area (proposal)"))
    ax.add_patch(Circle((0.20, 0.20), 0.05, fill=False, ec="tab:blue", lw=2, ls="--",
                        label="pot (candidate)"))
    ax.plot(0, 0, "ks")
    ax.set_xlim(-0.02, extent[1])
    ax.set_title(title)
    ax.set_xlabel("x [m] (robot forward)")
    ax.set_ylabel("y [m]")
    ax.set_aspect("equal")
axes[0].legend(loc="lower left", fontsize=8)
fig.savefig(OUT / "reachability.png", dpi=100, bbox_inches="tight")
plt.show()

in_area = (xs[None, :] >= 0.14) & (xs[None, :] <= 0.26) & (np.abs(ys[:, None]) <= 0.10)
coverage = float(grid_grasp[in_area].mean())
print(f"作業領域案（x 0.14〜0.26m, y ±0.10m）のうち真上把持できる割合: {coverage:.0%}")
summary["work_area_top_down_coverage"] = round(coverage, 3)

## 8. 小片（立方体）の把持テスト

真上把持の IK で経由点（上空 → 下降 → 閉じる → 持ち上げ）を作り、関節目標を補間して実行する
（段階1のエキスパートの原型）。1 辺 20 / 25 / 30mm の立方体を作業領域内のランダムな位置・向きに
置いて試し、成功率を測る。成功判定は so101-nexus の PickLift と同じ（掴んだまま 5cm 以上持ち上げる）。

In [ ]:
from so101_food.expert.simple_pick import run_pick_trial

N_TRIALS = 10
rows, video_frames = [], []
for side in (20, 25, 30):
    cfg_pick = PickConfig(
        objects=CubeObject(side_length_mm=side, color="orange"),
        render=RenderConfig(quality=QUALITY, camera="side"),
        terminate_on_success=False,
        **SPAWN,
    )
    env_pick = gym.make("MuJoCoPickLift-v1", config=cfg_pick, render_mode="rgb_array",
                        max_episode_steps=1000)
    env_pick.reset(seed=0)
    ik = TopDownIK(env_pick.unwrapped.model)
    for seed in range(N_TRIALS):
        result = run_pick_trial(env_pick, ik, seed=seed, render=(side == 25 and seed < 3))
        video_frames += result.pop("frames")[::2]
        rows.append({"side_mm": side, **result})
    env_pick.close()

df_pick = pd.DataFrame(rows)
df_pick.to_csv(OUT / "pick_trials.csv", index=False)
rates = df_pick.groupby("side_mm")["success"].mean()
display(rates.to_frame("success rate"))
summary["scripted_pick_success_rate"] = {int(k): float(v) for k, v in rates.items()}

path = save_mp4(video_frames, OUT / "scripted_pick.mp4", fps=25)
print("saved:", path.relative_to(REPO))
display(Video(str(path), embed=True, width=480))

## 9. 結果のまとめ

下に表示される JSON（`outputs/00_env_check/summary.json`）を共有してほしい。
動画は JupyterLab のファイルブラウザから `outputs/00_env_check/*.mp4` をダウンロードして確認できる。

In [ ]:
env_cam.close()
(OUT / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2))
print(json.dumps(summary, ensure_ascii=False, indent=2))